# 057 — original final solution

Rebuild the newer branches of **057_quality (Version 42, public 0.99978)**. Historical R50, thaw, sigft, teacher and train donors are frozen inputs. ConvNeXt 044, both U045 heads and both seeds of counters 053/054 are trained with their original settings. No cold-start reconstruction of the historical models.

Attach the competition and the private dataset made from **057.zip**. Select **T4 x2**, **Internet Off**, then **Save & Run All**. The notebook applies the original four-view R50 TTA, count mixture, hidden prior and guarded swap, and checks the result against the saved 057 CSV.


In [ ]:
import time
STARTED = time.monotonic()


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch
input_root = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("KAGGLE_WORK_ROOT", "/kaggle/working"))
projects = sorted({p.parent for p in input_root.rglob("snapshot057.json")
                   if (p.parent / "src").is_dir() and (p.parent / "configs").is_dir()})
assert len(projects) <= 1, f"Several final datasets found: {projects}"
if projects:
    project = projects[0]
else:
    archives = list(input_root.rglob("057.zip"))
    assert len(archives) == 1, "Attach the private dataset made from 057.zip"
    project = work_root / "project"
    project.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            assert (project / name).resolve().is_relative_to(project.resolve()), "Unsafe archive path"
        archive.extractall(project)
spec = json.loads((project / "snapshot057.json").read_text())
for name, digest in spec["files"].items():
    path = (project / name).resolve()
    assert path.is_relative_to(project.resolve()), "Unsafe source path"
    with path.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, f"Source checksum mismatch: {name}"
roots = sorted({p.parent for p in input_root.rglob("train.csv")
                if (p.parent / "train").is_dir() and (p.parent / "test").is_dir()
                and (p.parent / "test.csv").is_file()
                and (p.parent / "sample_submission.csv").is_file()})
assert len(roots) == 1, f"Attach one competition dataset: {roots}"
data = roots[0]
assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
sys.path.insert(0, str(project))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
print("057_quality | cached historical sources | train C044 / U045 / counters | T4 x2")


## Train the newer branches and run 057


In [ ]:
import numpy as np
import pandas as pd
from src.locked import run
report = run(project, data, Path("/kaggle/temp/locked057"), work_root, started=STARTED)
actual = pd.read_csv(work_root / "final_submission.csv", float_precision="round_trip")
reference = pd.read_csv(project / "reference/057_quality.csv", float_precision="round_trip")
assert np.array_equal(actual.id, reference.id), "Reference IDs differ"
error = float(np.max(np.abs(actual.label.to_numpy() - reference.label.to_numpy())))
changed = int(np.count_nonzero((actual.label > 0.5) != (reference.label > 0.5)))
print(f"Compared with original 057: max probability difference={error:.3g}, changed row decisions={changed}")
assert changed == 0 and error < 1e-8, "Training differs from original 057; retain selected Version42"
print(f"Total: {report['seconds'] / 60:.2f} minutes; within 50 minutes: {report['within_50_minutes']}")
print("Ready: final_submission.csv")
